<a href="https://colab.research.google.com/github/Khushikumari24/Open-PDF-Assistant/blob/main/Open_PDF_Assistant.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q langchain langchain-community faiss-cpu pypdf gradio transformers accelerate bitsandbytes sentence-transformers

In [2]:
import os
import gradio as gr
import torch

# --- FREE OPEN-SOURCE IMPORTS ---
# Notice: NO "import openai" anywhere. We are doing this locally!
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.llms import HuggingFacePipeline

from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline, BitsAndBytesConfig

print("✅ All imports successful! (No API keys required)")

# 1. GLOBAL VARIABLES
vector_store = None
llm = None

# 2. LOAD THE OPEN-SOURCE AI MODEL LOCALLY ON COLAB GPU
def load_model():
    # This is a free, open-source model. It downloads directly to your Colab session.
    model_id = "Qwen/Qwen2.5-1.5B-Instruct"

    # This compresses the model to fit on the FREE Colab T4 GPU
    quantization_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4"
    )

    # Download the model weights directly from Hugging Face (Free)
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        quantization_config=quantization_config,
        device_map="auto" # Automatically uses your Colab GPU
    )

    text_generation_pipeline = pipeline(
        "text-generation",
        model=model,
        tokenizer=tokenizer,
        max_new_tokens=512,
        temperature=0.1,
        return_full_text=False,
        pad_token_id=tokenizer.eos_token_id
    )

    return HuggingFacePipeline(pipeline=text_generation_pipeline)

print("Downloading and loading free open-source model to Colab GPU... (takes ~1 min)")
llm = load_model()
print("✅ Model loaded successfully! Running 100% locally, no API calls made.")

# 3. PROCESS THE UPLOADED PDF
def process_pdf(file_obj):
    global vector_store

    if file_obj is None:
        return "⚠️ Please upload a PDF file first."

    file_path = file_obj.name

    loader = PyPDFLoader(file_path)
    documents = loader.load()

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=500,
        chunk_overlap=50
    )
    chunks = text_splitter.split_documents(documents)

    # Create vector store locally in Colab's RAM
    embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
    vector_store = FAISS.from_documents(chunks, embeddings)

    return f"✅ Successfully processed: {os.path.basename(file_path)}. Ready to chat!"

# 4. THE MANUAL RAG CHAT LOGIC (Runs entirely on your Colab GPU)
def chat_with_pdf(message, history):
    global vector_store, llm

    if vector_store is None or llm is None:
        return history + [[message, "⚠️ Please upload and process a PDF first!"]]

    # Step A: Search the local vector database
    retrieved_docs = vector_store.similarity_search(message, k=3)
    context = "\n\n".join([doc.page_content for doc in retrieved_docs])

    # Step B: Format history
    history_text = ""
    if history:
        for user_msg, ai_msg in history:
            history_text += f"User: {user_msg}\nAssistant: {ai_msg}\n"

    # Step C: Build the prompt
    prompt = f"""You are a helpful enterprise AI assistant. Answer the user's question based ONLY on the provided context.
If the answer is not in the context, say "I don't know based on the provided document."

Context:
{context}

Chat History:
{history_text}

User Question: {message}

Assistant:"""

    # Step D: Get response from the LOCAL model
    try:
        response = llm.invoke(prompt)
        answer = response.strip()
    except Exception as e:
        answer = f"⚠️ An error occurred: {str(e)}"

    return history + [[message, answer]]

# 5. BUILD THE GRADIO UI
with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 📄 Free Open-Source PDF AI Assistant")
    gr.Markdown("100% Free. No API Keys. Runs locally on Google Colab GPU.")

    with gr.Row():
        with gr.Column(scale=1):
            pdf_input = gr.File(label="Upload PDF Document", file_types=[".pdf"])
            process_btn = gr.Button("Process Document", variant="primary")
            status_output = gr.Textbox(label="Status", interactive=False)

        with gr.Column(scale=2):
            chatbot = gr.Chatbot(label="Chat with your PDF", height=400)
            msg_input = gr.Textbox(label="Ask a question...", placeholder="e.g., What is the policy?")
            clear_btn = gr.ClearButton([msg_input, chatbot])

    process_btn.click(fn=process_pdf, inputs=[pdf_input], outputs=[status_output])
    msg_input.submit(fn=chat_with_pdf, inputs=[msg_input, chatbot], outputs=[chatbot])

# 6. LAUNCH
print("Starting Gradio interface...")
demo.launch(share=True, debug=True)

/tmp/ipykernel_3108/3330901475.py:7: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


✅ All imports successful! (No API keys required)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize

Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://36ef764aba152cc1ef.gradio.live


How it Works (The Engineering Logic)
Ingestion: PDF is loaded and split into 500-character overlapping chunks.

Embedding: Chunks are converted to vectors using a lightweight embedding model and stored in a local FAISS index.

Retrieval: User queries are embedded and matched against the FAISS index to find the top 3 most relevant chunks
.
Generation: The retrieved context and chat history are injected into a strict system prompt, guiding the quantized Qwen model to generate an accurate, hallucination-free response.

In [5]:

#### File 3: `app.py`

import os
import gradio as gr
import torch

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.llms import HuggingFacePipeline

from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline, BitsAndBytesConfig

vector_store = None
llm = None

def load_model():
    model_id = "Qwen/Qwen2.5-1.5B-Instruct"
    quantization_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4"
    )
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        quantization_config=quantization_config,
        device_map="auto"
    )
    text_generation_pipeline = pipeline(
        "text-generation", model=model, tokenizer=tokenizer,
        max_new_tokens=512, temperature=0.1, return_full_text=False,
        pad_token_id=tokenizer.eos_token_id
    )
    return HuggingFacePipeline(pipeline=text_generation_pipeline)

print("Loading model...")
llm = load_model()
print("✅ Model loaded!")

def process_pdf(file_obj):
    global vector_store
    if file_obj is None: return "⚠️ Please upload a PDF file first."

    loader = PyPDFLoader(file_obj.name)
    documents = loader.load()
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
    chunks = text_splitter.split_documents(documents)

    embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
    vector_store = FAISS.from_documents(chunks, embeddings)
    return f"✅ Processed: {os.path.basename(file_obj.name)}. Ready to chat!"

def chat_with_pdf(message, history):
    global vector_store, llm
    if vector_store is None or llm is None:
        return history + [[message, "⚠️ Please upload and process a PDF first!"]]

    retrieved_docs = vector_store.similarity_search(message, k=3)
    context = "\n\n".join([doc.page_content for doc in retrieved_docs])

    history_text = ""
    if history:
        for user_msg, ai_msg in history:
            history_text += f"User: {user_msg}\nAssistant: {ai_msg}\n"

    prompt = f"""You are a helpful enterprise AI assistant. Answer based ONLY on the provided context.
If the answer is not in the context, say "I don't know based on the provided document."

Context:
{context}

Chat History:
{history_text}

User Question: {message}
Assistant:"""

    try:
        response = llm.invoke(prompt)
        answer = response.strip()
    except Exception as e:
        answer = f"⚠️ Error: {str(e)}"

    return history + [[message, answer]]

with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 📄 Open PDF Assistant")
    gr.Markdown("100% Free. No API Keys. Runs locally.")
    with gr.Row():
        with gr.Column(scale=1):
            pdf_input = gr.File(label="Upload PDF", file_types=[".pdf"])
            process_btn = gr.Button("Process Document", variant="primary")
            status_output = gr.Textbox(label="Status", interactive=False)
        with gr.Column(scale=2):
            chatbot = gr.Chatbot(label="Chat", height=400)
            msg_input = gr.Textbox(label="Ask a question...", placeholder="e.g., What is the policy?")
            clear_btn = gr.ClearButton([msg_input, chatbot])

    process_btn.click(fn=process_pdf, inputs=[pdf_input], outputs=[status_output])
    msg_input.submit(fn=chat_with_pdf, inputs=[msg_input, chatbot], outputs=[chatbot])

if __name__ == "__main__":
    demo.launch(share=True)

Loading model...


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


✅ Model loaded!


/tmp/ipykernel_3108/1252156413.py:89: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:
/tmp/ipykernel_3108/1252156413.py:98: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot(label="Chat", height=400)
/tmp/ipykernel_3108/1252156413.py:98: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to disable tags in your chatbot.
  chatbot = gr.Chatbot(label="Chat", height=400)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://7bf29f3fe03a3cd24a.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
